In [ ]:
import pandas as pd
import numpy as np
import regex as rex

import emoji
import re

from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, models

In [ ]:
import torch
torch.cuda.is_available()
# True bedeutet GPU verfügbar
torch.cuda.current_device(), torch.cuda.get_device_name(0)

(0, 'Tesla T4')

In [ ]:
#model = SentenceTransformer('all-mpnet-base-v2')  # schnell + leicht
word_embedding_model = models.Transformer("vinai/bertweet-base")
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model])#, device="cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/bertweet-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
chunks = pd.read_csv(
    "bitcoin_tweets_clean_bert.csv",
    chunksize=100_000,
    engine="python",          # 🔴 extrem wichtig
)

In [ ]:
df = pd.concat(chunks)

In [ ]:
chunks_to_process = list(range(0, 39))#list(range(0, 67))

In [ ]:
batch_size = 16
chunk_size = 100 # anpassen je nach RAM

In [ ]:
from pathlib import Path
for i, start in enumerate(range(0, len(df), chunk_size)):
    if i not in chunks_to_process:
      continue

    chunk_path = Path(f'embeddings_chunk_{i}.parquet')
    if chunk_path.exists():
        print(f"Chunk {i} existiert schon, überspringe...")
        continue

    chunk = df.iloc[start:start+chunk_size].copy()
    # Embeddings erzeugen
    embeddings = model.encode(chunk['text'].tolist(),
                              batch_size=batch_size,
                              convert_to_numpy=True,
                              normalize_embeddings=True,
                              show_progress_bar=True)

    # In Float16 speichern, spart RAM und Platz
    embeddings = embeddings.astype(np.float16)
    emb_cols = [f"emb_{j}" for j in range(embeddings.shape[1])]
    chunk_embeddings_df = pd.DataFrame(embeddings, index=chunk.index, columns=emb_cols)
    chunk_embeddings_df['username']= chunk['username'].values
    chunk_embeddings_df['datetime']= chunk['datetime'].values

    # Chunk als Parquet speichern
    chunk_embeddings_df.to_parquet(f'embeddings_chunk_{i}.parquet', compression='brotli')

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

KeyboardInterrupt: 